# NELA-GT Deep Learning Training — Colab A100

Trains **BiLSTM** and **TextCNN** (finetuned only) on the NELA-GT 500k 3-class dataset.

**Classes:** 0=Reliable | 1=Questionable | 2=Conspiracy

**Before running:** upload the required files to Google Drive (see Cell 2).

## Files to upload to Google Drive

Upload everything to a folder called `news-classification` in your Drive root.

```
MyDrive/
└── news-classification/
    ├── data/
    │   ├── processed/
    │   │   └── nela_sampled_500k/
    │   │       ├── train.csv          (~1.2 GB)
    │   │       ├── val.csv            (~150 MB)
    │   │       └── test.csv           (~150 MB)
    │   └── raw/
    │       └── glove/
    │           └── glove.840B.300d.txt  (2.1 GB)
    └── src/                           (zip and upload as src.zip)
```

**Local paths on your machine:**
- Processed data: `~/news-classification/data/processed/nela_sampled_500k/`
- GloVe: `~/news-classification/data/raw/glove/glove.840B.300d.txt`
- Source code: `~/news-classification/src/` → zip as `src.zip`

**Total upload size: ~3.6 GB**

To zip the src folder, run on your local machine:
```bash
cd ~/news-classification
zip -r src.zip src/
```

In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2 — Setup: unzip src, install dependencies, set paths
import os, sys

DRIVE_ROOT = '/content/drive/MyDrive/news-classification'
WORK_DIR   = '/content/news-classification'

os.makedirs(WORK_DIR, exist_ok=True)

# Unzip source code
!unzip -q "{DRIVE_ROOT}/src.zip" -d "{WORK_DIR}"

# Symlink data so paths resolve identically to local machine
data_link = f'{WORK_DIR}/data'
if not os.path.exists(data_link):
    os.symlink(f'{DRIVE_ROOT}/data', data_link)

# Results + checkpoints go to Drive so they survive session end
for d in ['results/runs', 'checkpoints/deep_learning', 'data/cache', 'logs']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)

results_link = f'{WORK_DIR}/results'
if not os.path.exists(results_link):
    os.symlink(f'{DRIVE_ROOT}/results', results_link)

ckpt_link = f'{WORK_DIR}/checkpoints'
if not os.path.exists(ckpt_link):
    os.symlink(f'{DRIVE_ROOT}/checkpoints', ckpt_link)

logs_link = f'{WORK_DIR}/logs'
if not os.path.exists(logs_link):
    os.symlink(f'{DRIVE_ROOT}/logs', logs_link)

# Add project root to path
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')
print(f'Python path includes: {WORK_DIR}')

In [ ]:
# Cell 3 — Install dependencies
!pip install -q nltk scikit-learn pandas numpy torch torchvision

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 4 — Verify all required files are present
import os

required = [
    'data/processed/nela_sampled_500k/train.csv',
    'data/processed/nela_sampled_500k/val.csv',
    'data/processed/nela_sampled_500k/test.csv',
    'data/raw/glove/glove.840B.300d.txt',
]

all_ok = True
for f in required:
    size = os.path.getsize(f) / 1e6 if os.path.exists(f) else -1
    status = f'✓  {size:.0f} MB' if size > 0 else '✗  MISSING'
    print(f'{status}  {f}')
    if size < 0:
        all_ok = False

if all_ok:
    print('\nAll files present — ready to train.')
else:
    print('\n⚠ Some files are missing. Check your Drive upload.')

In [ ]:
# Cell 5 — Load dataset splits
from src.utils.config import NELA_DL_CONFIG
from src.utils.data_loader import load_splits

train_df, val_df, test_df = load_splits(NELA_DL_CONFIG)
num_classes = len(NELA_DL_CONFIG.label_map)

print(f'Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
print(f'Classes: {num_classes}')
print('Label distribution (train):')
label_names = {0: 'Reliable', 1: 'Questionable', 2: 'Conspiracy'}
for lbl, cnt in train_df['label'].value_counts().sort_index().items():
    print(f'  {lbl} ({label_names[lbl]}): {cnt:,} ({100*cnt/len(train_df):.1f}%)')

In [ ]:
# Cell 6 — Build vocabulary (cached to Drive on first run)
import os
from src.deep_learning.dataset import VocabBuilder

dataset_name = 'nela_sampled_500k'
vocab_cache = f'data/cache/vocab_{dataset_name}.json'

vocab_builder = VocabBuilder()
if os.path.exists(vocab_cache):
    print('Loading vocab from cache...')
    vocab = VocabBuilder.load(dataset_name)
else:
    print('Building vocab from training set (this takes ~5 min)...')
    vocab = vocab_builder.build(train_df['text'].tolist())
    vocab_builder.save(vocab, dataset_name)

print(f'Vocab size: {len(vocab):,}')

In [ ]:
# Cell 7 — Load GloVe embeddings (takes ~5 min, loads 2.2M vectors)
from pathlib import Path
from src.deep_learning.embeddings import GloVeLoader
from src.utils.config import GLOVE_DIM

GLOVE_PATH = Path('data/raw/glove/glove.840B.300d.txt')
glove = GloVeLoader(GLOVE_PATH, dim=GLOVE_DIM)

# Pre-load vectors into memory now — shared across all model runs
glove._load()
print('GloVe loaded.')

In [ ]:
# Cell 8 — Training configuration
# Adjust batch_size if you run out of VRAM (A100 80GB can handle 256 comfortably)

EPOCHS     = 15
BATCH_SIZE = 256   # A100: 256; if OOM reduce to 128
MAX_SEQ_LEN = 512

ARCHS        = ['textcnn', 'bilstm']   # TextCNN first (faster), BiLSTM second
FREEZE_MODES = ['finetuned']           # finetuned only — frozen adds little value at 500k

print(f'Epochs: {EPOCHS}  |  Batch size: {BATCH_SIZE}  |  Seq len: {MAX_SEQ_LEN}')
print(f'Models: {ARCHS}  |  Freeze modes: {FREEZE_MODES}')
print(f'Estimated time — TextCNN: ~25 min  |  BiLSTM: ~90 min')

In [ ]:
# Cell 9 — Training loop
import torch
from src.deep_learning.dataset import build_dataloaders
from src.deep_learning.models import BiLSTM, TextCNN
from src.deep_learning.trainer import Trainer, TrainerConfig
from src.utils.metrics import evaluate
from src.utils.seeds import set_seed

set_seed(42)

all_results = []

for arch in ARCHS:
    for freeze_mode in FREEZE_MODES:
        freeze = (freeze_mode == 'frozen')
        model_name = f'{arch}_{freeze_mode}'
        print(f'\n{"="*60}')
        print(f'Training: {model_name.upper()} on {dataset_name}')
        print(f'{"="*60}')

        train_loader, val_loader, test_loader = build_dataloaders(
            train_df, val_df, test_df, vocab,
            max_len=MAX_SEQ_LEN,
            batch_size=BATCH_SIZE,
        )

        embedding = glove.build_matrix(vocab, freeze=freeze)

        if arch == 'bilstm':
            model = BiLSTM(embedding, num_classes=num_classes)
        else:
            model = TextCNN(embedding, num_classes=num_classes)

        trainer_cfg = TrainerConfig(epochs=EPOCHS, batch_size=BATCH_SIZE)
        trainer = Trainer(model, trainer_cfg, model_name, dataset_name)

        history = trainer.train(train_loader, val_loader)

        # Reload best checkpoint
        import os
        from src.utils.config import CHECKPOINTS_PATH
        ckpt_path = os.path.join(CHECKPOINTS_PATH, 'deep_learning', dataset_name, f'{model_name}.pt')
        model.load_state_dict(torch.load(ckpt_path, map_location=trainer_cfg.device))
        model.to(trainer_cfg.device).eval()

        def predict_fn(texts, _model=model, _vocab=vocab, _dev=trainer_cfg.device):
            from src.deep_learning.dataset import TextDataset
            import numpy as np
            ds = TextDataset(texts, [0]*len(texts), _vocab, MAX_SEQ_LEN)
            all_preds, all_probas = [], []
            with torch.no_grad():
                for i in range(0, len(ds), BATCH_SIZE):
                    batch_ids = torch.stack(
                        [ds[j]['input_ids'] for j in range(i, min(i+BATCH_SIZE, len(ds)))]
                    ).to(_dev)
                    logits = _model(batch_ids)
                    probas = torch.softmax(logits, dim=-1).cpu().numpy()
                    all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
                    all_probas.append(probas)
            return np.array(all_preds), np.vstack(all_probas)

        hyperparams = {
            'arch': arch, 'freeze_mode': freeze_mode,
            'epochs_trained': history.best_epoch,
            'learning_rate': trainer_cfg.learning_rate,
            'batch_size': BATCH_SIZE, 'max_seq_len': MAX_SEQ_LEN,
            'glove_dim': GLOVE_DIM, 'vocab_size': len(vocab),
        }

        bundle = evaluate(
            predict_fn=predict_fn,
            test_df=test_df,
            model_name=model_name,
            dataset_name=dataset_name,
            train_time_sec=history.train_time_sec,
            hyperparams=hyperparams,
        )
        all_results.append(bundle)
        print(f'  acc={bundle.accuracy:.4f}  macro_f1={bundle.macro_f1:.4f}  auc={bundle.roc_auc:.4f}')

print('\nAll models trained.')

In [ ]:
# Cell 10 — Results summary
import pandas as pd

rows = []
for b in all_results:
    rows.append({
        'Model':      b.model_name,
        'Accuracy':   round(b.accuracy, 4),
        'Macro F1':   round(b.macro_f1, 4),
        'ROC-AUC':    round(b.roc_auc, 4),
        'Train (s)':  round(b.train_time_sec, 1),
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

In [ ]:
# Cell 11 — Per-class metrics
label_names = {0: 'Reliable', 1: 'Questionable', 2: 'Conspiracy'}

for b in all_results:
    print(f'\n--- {b.model_name} ---')
    print(f'{"Class":<16} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
    print('-' * 56)
    for i, name in label_names.items():
        p = b.per_class_metrics['precision'][i]
        r = b.per_class_metrics['recall'][i]
        f = b.per_class_metrics['f1'][i]
        s = b.per_class_metrics['support'][i]
        print(f'{name:<16} {p:>10.4f} {r:>10.4f} {f:>10.4f} {s:>10,}')

In [ ]:
# Cell 12 — Confirm output files saved to Drive
import os, glob

print('Checkpoints saved:')
for f in glob.glob(f'{DRIVE_ROOT}/checkpoints/deep_learning/{dataset_name}/*.pt'):
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f)}  ({size_mb:.0f} MB)')

print('\nResult JSONs saved:')
for f in sorted(glob.glob(f'{DRIVE_ROOT}/results/runs/*{dataset_name}*.json')):
    print(f'  {os.path.basename(f)}')

print('\nVocab cache:')
vc = f'{DRIVE_ROOT}/data/cache/vocab_{dataset_name}.json'
if os.path.exists(vc):
    print(f'  vocab_{dataset_name}.json  ({os.path.getsize(vc)/1e6:.0f} MB)')

print('\nAll outputs are persisted to Drive — safe to close Colab session.')